# Extrusion Tutorial, as seen in SV Paper
- This tutorial will walk you through how to run a full inversion MiChroM training with the SV paper implementation of extrusion
- You can change the functions in OpenMiChroM/Extrusion.py to adjust the parameters and assumptions of the extrusion model
- Indeed, one need not use an extrusion model at all; this notebook can be used as a guide for training a Full Inversion potential with a Hamiltonian that changes throughout the simulation trajectory by repeatedly re-initializing the `simulation` object with a new potential energy function 
- Here, you will optimize the effective potential matrix of a 340-bead (at 10kb bins = 3.4 Mb) stretch of the *epha4* locus of the mouse genome 
### We won't run a full optimization step here. For details on this procedure, see Full Inversion documentation. This notebook is meant to demonstrate the simulation side of a dynamic-Hamiltonian Full Inversion training, which can be slightly reworked and plugged in to a Full Inversion optimization procedure.

In [10]:
# import packages

import numpy as np
import os
from pathlib import Path
import h5py


from OpenMiChroM.ChromDynamics import MiChroM # OpenMiChroM simulation module
from OpenMiChroM.Extrusion_Bonds import Loop_Extrusion_Manager # generates extrusion bonds list
from OpenMiChroM.Optimization import AdamTraining # Adam optimizer for Full Inversion potential

In [11]:
# adjust path 

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "setup.py").exists() and (path / "OpenMiChroM").is_dir():
        os.chdir(path)
        break
else:
    raise FileNotFoundError("Could not find the OpenMiChroM repo root.")

print(f"Current directory = \n{os.getcwd()}")

Current directory = 
/Users/milesgantcher/Library/CloudStorage/OneDrive-RiceUniversity/Documents/VSCode_Projects/Onuchic_VS_Code/OpenMiChroM


Here, we're training on a contact map from a mouse near the *epha4* locus. This data is from Bianco et al., 2018.

Further, we load motif tracks; these are by bead, with each entry = the probability that an extruder foot becomes fixed upon contact. These probabilities were generated via the procedure described in the SV paper from data found in Andrey et al., 2017.

Finally, we have the path to a blank lambda matrix which we would train to be our effective potential matrix if we were running a complete Full Inversion step. 

In [12]:
locus_start = 440
locus_end = 780
fs_fix_probs = np.load('Tutorials/Structural_Variations/Data/Simulation_Inputs/forwards_fix_probs.npy')[locus_start:locus_end]
rs_fix_probs = np.load('Tutorials/Structural_Variations/Data/Simulation_Inputs/reverse_fix_probs.npy')[locus_start:locus_end]
exp_map = np.load('Tutorials/Structural_Variations/Data/Simulation_Inputs/mouse_WT_small.npy')

blank_lambdas = 'Tutorials/Structural_Variations/Data/Simulation_Inputs/blank_lambdas.csv'

chrom_length = exp_map.shape[0]

print(f"Modeling a locus of size {chrom_length} x {chrom_length}")

Modeling a locus of size 340 x 340


Timing parameters for the simulation. In the SV paper, we use `initial_run_steps = 5*10**5`, `timesteps_per_extrude = num_extrusion_steps = report_interval = 10**3`. These parameters are not physically motivated, but instead set to maximize optimization speed without adding noise. `report_interval` sets the interval at which MiChroM's reporters save the state (eg. to a .cndb file).

In [13]:
# Length of initial run, which adds a flat bottom harmonic potential to collapse the initial structure
initial_run_steps = 3*10**4
# Number of integrator timesteps per extrusion step
timesteps_per_extrude = 10**2
# Number of extrusion steps
num_extrusion_steps = 10**2
print(f"Running for {num_extrusion_steps * timesteps_per_extrude + initial_run_steps} total timesteps.")

report_interval = 10**2

Running for 40000 total timesteps.


Extruder density is carried over from SV paper. 

In [14]:
extruder_density = 6/100
extruder_count = int(chrom_length * extruder_density)
print(f"Extruders at a density of {extruder_density*100} per 100 beads for a total of {extruder_count} motors in this simluation.")


Extruders at a density of 6.0 per 100 beads for a total of 20 motors in this simluation.


Generate extruder trajectories

In [15]:
manager = Loop_Extrusion_Manager(
        fprobs_fix=fs_fix_probs,
        rprobs_fix=rs_fix_probs,
        num_steps=num_extrusion_steps,
        extruder_count=extruder_count,
        )
loop_trajs = manager.get_extrusion_bonds()
print(f"Trajectory of the first extruder = {loop_trajs[:,0,:]}")

Generated extrusion bonds list. 
Trajectory of the first extruder = [[160 162]
 [159 163]
 [158 164]
 [157 165]
 [156 166]
 [155 167]
 [154 168]
 [153 169]
 [152 170]
 [151 171]
 [150 172]
 [149 173]
 [148 174]
 [147 175]
 [146 176]
 [145 177]
 [144 178]
 [143 179]
 [142 180]
 [141 181]
 [140 182]
 [139 183]
 [138 184]
 [137 185]
 [136 186]
 [135 187]
 [134 188]
 [133 189]
 [132 190]
 [131 191]
 [130 192]
 [129 193]
 [128 194]
 [127 195]
 [126 196]
 [125 197]
 [124 198]
 [123 199]
 [122 200]
 [121 201]
 [120 202]
 [119 203]
 [118 204]
 [117 205]
 [116 206]
 [115 207]
 [114 208]
 [113 209]
 [112 210]
 [111 211]
 [110 212]
 [109 213]
 [108 214]
 [107 215]
 [106 216]
 [105 217]
 [104 218]
 [103 219]
 [102 220]
 [101 221]
 [100 222]
 [ 99 223]
 [ 98 224]
 [ 97 225]
 [ 96 226]
 [ 95 227]
 [ 94 228]
 [ 93 229]
 [ 92 230]
 [ 91 231]
 [ 90 232]
 [ 89 233]
 [ 88 234]
 [ 87 235]
 [ 86 236]
 [ 85 237]
 [ 84 238]
 [ 83 239]
 [ 82 240]
 [ 81 241]
 [ 80 242]
 [ 79 243]
 [ 78 244]
 [ 77 245]
 [ 76 24

For details on MiChroM, see other tutorials in this repo. Here, we initialize a MiChroM object, assign it computing resources, and build an initial structure.

In [16]:
seq_file = 'Tutorials/Structural_Variations/Data/Simulation_Inputs/mouse.seq'

test_sim = MiChroM(name='sim_0', temperature=1.0, timeStep=0.01,printing=True)
test_sim.setup(platform="cpu",printing=True)
test_sim.saveFolder('Tutorials/Structural_Variations/Data/Simulation_Data')
initial_struct = test_sim.createSpringSpiral(ChromSeq=seq_file,isRing=False)
test_sim.loadStructure(initial_struct, center=True)


    ***************************************************************************************     
     **** **** *** *** *** *** *** *** OpenMiChroM-1.1.1 *** *** *** *** *** *** **** ****      

         OpenMiChroM is a Python library for performing chromatin dynamics simulations.         
                            OpenMiChroM uses the OpenMM Python API,                             
                employing the MiChroM (Minimal Chromatin Model) energy function.                
      The chromatin dynamics simulations generate an ensemble of 3D chromosomal structures      
      that are consistent with experimental Hi-C maps, also allows simulations of a single      
                 or multiple chromosome chain using High-Performance Computing                  
                            in different platforms (GPUs and CPUs).                             

         OpenMiChroM documentation is available at https://open-michrom.readthedocs.io          

         OpenMiChroM is des

Here we add potential energies to our initial simulation, including a flat bottom harmonic term that will collapse our chromatin to a useable volume. 

In [17]:
test_sim.buildInitialCollapseSim(lambdaFile=blank_lambdas)

FENEBond was added
AngleForce was added
RepulsiveSoftCore was added
CustomTypes was added
FlatBottomHarmonic was added
Setting positions... loaded!
Setting velocities... loaded!
Context created!

Simulation name: sim_0
Number of beads: 340, Number of chains: 1
Potential energy: 20.66615, Kinetic Energy: 1.48861 at temperature: 1.0

Potential energy per forceGroup:
                                Values
FENEBond                  6918.078176
AngleForce                   7.876326
RepulsiveSoftCore            0.000000
CustomTypes                  0.000000
FlatBottomHarmonic         100.535906
Potential Energy (total)  7026.490407


Add reporters. See MiChroM documentation for details.

In [18]:
test_sim.createReporters(statistics=False, traj=True, trajFormat="cndb", energyComponents=False, 
                         interval=report_interval)

Run for the inital collapse. 

In [19]:
test_sim.run(nsteps=initial_run_steps, report=True)

#"Progress (%)"	"Step"	"Speed (ns/day)"	"Time Remaining"
33.3%	10000	0	--
66.7%	20000	2.02e+03	0:04
100.0%	30000	2.05e+03	0:00


Save the position and velocity vectors of the simulation. 

In [20]:
curr_struct = test_sim.saveStructure(mode='gro_like')
curr_vels = test_sim.get_velocities()

Set up our Adam optimizer for training the pairwise potential matrix. 

In [21]:
opt = AdamTraining(mu=2.5, rc=2.0, eta=0.01, it=0, updateNeeded=True)
opt.getHiCexp(HiC='Tutorials/Structural_Variations/Data/Simulation_Inputs/mouse_WT_small.npy',norm=False, cutoff_low=0.001,neighbors=2)

Here, we run our extended simulation.

In [22]:
for t in range(1,num_extrusion_steps):
    if t%(num_extrusion_steps/100) == 0:
        print(f"Now running step {t} out of {num_extrusion_steps}")

    # set up a subsequent simulation
    test_sim = MiChroM(name=f'sim_{t}', temperature=1.0, timeStep=0.01, printing = False)
    test_sim.setup(platform="cpu", printing = False)
    test_sim.saveFolder('Tutorials/Structural_Variations/Data/Simulation_Data')
    
    # add the relevant potential energies, including the extruder potentials. Don't add a flat bottom harmonic.
    test_sim.buildSubsequentExtrusionSim(chromosome='chr1',loop_list=loop_trajs[t-1],CoordFiles = curr_struct,Velocities = curr_vels,lambdaFile=blank_lambdas)
    # create reporters
    test_sim.createReporters(statistics=False,traj=True, trajFormat="cndb",interval=report_interval)
    # run the simulation
    test_sim.run(nsteps=timesteps_per_extrude, report=False)
    # save the final state
    curr_struct = test_sim.saveStructure(mode='gro_like')
    curr_vels = test_sim.get_velocities()
    # update the Adam optimizer. 
    opt.probCalc(test_sim.getPositions())

Now running step 1 out of 100
Now running step 2 out of 100
Now running step 3 out of 100
Now running step 4 out of 100
Now running step 5 out of 100
Now running step 6 out of 100
Now running step 7 out of 100
Now running step 8 out of 100
Now running step 9 out of 100
Now running step 10 out of 100
Now running step 11 out of 100
Now running step 12 out of 100
Now running step 13 out of 100
Now running step 14 out of 100
Now running step 15 out of 100
Now running step 16 out of 100
Now running step 17 out of 100
Now running step 18 out of 100
Now running step 19 out of 100
Now running step 20 out of 100
Now running step 21 out of 100
Now running step 22 out of 100
Now running step 23 out of 100
Now running step 24 out of 100
Now running step 25 out of 100
Now running step 26 out of 100
Now running step 27 out of 100
Now running step 28 out of 100
Now running step 29 out of 100
Now running step 30 out of 100
Now running step 31 out of 100
Now running step 32 out of 100
Now running step 

At the end of the loop, use the `opt` instance of the `AdamTraining` class to save the data from this run. 

In [23]:
print(f"creating Pi files")

print(f"test_sim.folder = {test_sim.folder}")

print(f"Pi = {opt.Pi}")

with h5py.File(test_sim.folder + "/Pi.h5", 'w') as hf:
    hf.create_dataset("Pi",  data=opt.Pi)

print(f"creating NFrames files")

with h5py.File(test_sim.folder + "/NFrames.h5", 'w') as hf:
    hf.create_dataset("NFrames",  data=opt.NFrames)

print(f"Done!")

creating Pi files
test_sim.folder = Tutorials/Structural_Variations/Data/Simulation_Data
Pi = [[9.89955056e+01 9.83818328e+01 7.73962709e+01 ... 1.32565381e-09
  2.45831433e-09 5.13149322e-08]
 [9.83818328e+01 9.89955056e+01 9.83436144e+01 ... 2.39431508e-09
  8.89582630e-10 2.86928787e-08]
 [7.73962709e+01 9.83436144e+01 9.89955056e+01 ... 4.16356616e-10
  2.20903573e-10 1.31761895e-08]
 ...
 [1.32565381e-09 2.39431508e-09 4.16356616e-10 ... 9.89955056e+01
  9.83295422e+01 7.50554420e+01]
 [2.45831433e-09 8.89582630e-10 2.20903573e-10 ... 9.83295422e+01
  9.89955056e+01 9.82862068e+01]
 [5.13149322e-08 2.86928787e-08 1.31761895e-08 ... 7.50554420e+01
  9.82862068e+01 9.89955056e+01]]
creating NFrames files
Done!
